# 16 — Azure Foundry and the platform landscape

You have now built the loop, given it tools, moved those tools to another process, wrapped the same desk in two frameworks, retrieved a corpus, guarded a ticket, and printed a bill.

A vendor is not selling you "an AI agent". They are selling one or more of **six layers** you have already written by hand. This chapter names the layers, says which ones are open standards, and says where lock-in actually lives.

Microsoft Foundry is the one managed path in this course, because Azure is the one cloud. **A naming note before you search for any of this:** the product was Azure AI Foundry and is now **Microsoft Foundry**. The older Agent Service surface is documented as *Foundry (classic)*. You will meet all three names in Microsoft's own docs; they are the same product line. AWS and Google sell the same six layers under different names. One sentence of orientation, then we stay here. No student needs an Azure credential. If the instructor did not provision a project, the captured shape is the lesson.


## 1. Learn

```
08  you wrote the desk
09  you drew the arrows
15  you scored one run
16  you are here — who sells each layer, and what you give up
17  whether any of it should exist
```

**Your notebook vs a managed runtime**

```mermaid
flowchart LR
    subgraph Yours["your notebook"]
        A["question"] --> B["your for-loop"]
        B --> C["your tools"]
    end
    subgraph Theirs["Foundry Agent Service"]
        D["question"] --> E["their runtime"]
        E --> F["their tool catalog"]
    end
```

The picture on the left is modules 03 to 15. The picture on the right is the same loop, as a product. The question for a buying meeting is not "should we use agents?" It is "which of these six layers are we buying, and which do we already have?"

| Layer | Problem it solves | The open pattern | What a cloud sells you | You already built |
|---|---|---|---|---|
| Model | Reasoning and tool selection | Any compatible endpoint | A managed model catalogue | Every `create` since 00 |
| Tools | Hands, portably | **MCP** | A governed tool catalogue | 02, then 06 |
| Orchestration | Control flow, state, retries | Open frameworks | A managed agent runtime | 03, then 08, then 09 |
| Context | Memory and grounding | Open vector stores | A managed search or memory service | the `messages` list; Chroma in 11 |
| Delegation | Agents talking to agents | **A2A** | A connected-agents feature | 13 |
| Governance | Identity, safety, evals, cost | OpenTelemetry, open evals | Identity, filters, hosted traces | 14 and 15 |

The two bold rows are open standards. Everything in the "cloud sells you" column is a vendor's implementation of a pattern you can build yourself. That is why the course built the pattern first.

AWS sells the same six as Bedrock / AgentCore. Google sells them as Vertex / Agent Engine. The names move. The layers do not.

Vendors also sort by **verb**, not by logo:

| Who | Verb | What you are buying |
|---|---|---|
| Non-technical | **use** | A finished application. Someone else chose the model, the tools, and the loop. |
| Non-technical | **build** | A visual studio. You configure an agent without writing the loop — and inherit whatever the vendor did not anticipate. |
| Technical | **develop** | The loop is yours. That is where these two days live. |
| Technical | **execute** | Managed infrastructure that runs a loop you wrote. This is where portability stops. |

These two days are **develop**. Foundry Agent Service is **execute**, with a **build** surface (Agent Builder) next to it. A graph framework you can also buy hosted sits in two columns at once. That is the business model, not sloppiness.

**Where lock-in lives**

```mermaid
flowchart TD
    P["portable"] --> M["model"]
    P --> T["tools behind MCP"]
    P --> O["open orchestration"]
    S["sticky"] --> Mem["memory"]
    S --> I["identity"]
    S --> Obs["observability"]
```

- **Portable:** the model (a compatible interface and a re-run of your evals). Tools, if they sit behind MCP. Orchestration, if you stayed on an open framework.
- **Sticky:** memory, identity, and the audit trail. Migrating a vector store is annoying. Migrating an identity model and a year of traces is a project.

Nobody argues about the model any more. The arguments start about eighteen months in, in the sticky column.

**What we will not do.** The classic Foundry surface was threads, runs, and messages. It is deprecated (retirement 31 March 2027). The current surface is **conversations** and **responses**, with a prompt-agent definition (`model`, `instructions`, `tools`). We do not install an Azure SDK and we do not call either API from this notebook. The lesson is the mapping, not a second client.

Cut first if the room is behind: the optional portal demo. Keep the six-layer table and the lock-in split.


## 2. Do

### Load the environment

Same load as every other module. The OpenAI key is unused here. The two Foundry names are optional. Neither value is printed.


In [1]:
from pathlib import Path
import os
from urllib.parse import urlparse

from dotenv import load_dotenv, find_dotenv


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
foundry_endpoint = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT", "").strip()
foundry_key = os.environ.get("AZURE_FOUNDRY_API_KEY", "").strip()

print("OPENAI_API_KEY is set:", bool(api_key))
print("AZURE_FOUNDRY_PROJECT_ENDPOINT is set:", bool(foundry_endpoint))
print("AZURE_FOUNDRY_API_KEY is set:", bool(foundry_key))
print("ROOT:", ROOT)
if foundry_endpoint:
    print("Foundry host:", urlparse(foundry_endpoint).netloc)


OPENAI_API_KEY is set: True
AZURE_FOUNDRY_PROJECT_ENDPOINT is set: False
AZURE_FOUNDRY_API_KEY is set: False
ROOT: /Users/tarekatwan/Downloads/ai_agents_course


### A managed desk, as a captured definition

This is the Chinook desk from module 08, written the way Foundry stores a **prompt agent**: a name, a model deployment, instructions, and a tool list. It is a teaching fixture, not a live create call. The field names match the current `PromptAgentDefinition` shape (`model`, `instructions`, `tools`). The invoke shape matches the current surface (`conversations` + `responses`), not threads and runs.


In [2]:
CAPTURED_AGENT = {
    "name": "chinook-desk",
    "kind": "prompt",
    "model": "<a Foundry model deployment>",
    "instructions": "Use the tools. Do not invent numbers or names.",
    "tools": [
        {"name": "invoice_count", "description": "How many invoices a customer has."},
        {"name": "invoice_total", "description": "What a customer has spent."},
        {"name": "support_rep", "description": "Who the customer's support representative is."},
    ],
}

CAPTURED_INVOKE = {
    "surface": "responses",
    "conversation": "conv_helena_1",
    "input": (
        "How many invoices does Helena Holy have, "
        "what is her total spend, and who is her support representative?"
    ),
    "output_text": (
        "Helena Holy has 7 invoices totaling $49.62. "
        "Her support representative is Steve Johnson."
    ),
}

print("agent name:   ", CAPTURED_AGENT["name"])
print("kind:         ", CAPTURED_AGENT["kind"])
print("model:        ", CAPTURED_AGENT["model"])
print("instructions: ", CAPTURED_AGENT["instructions"])
print("tools:        ", [t["name"] for t in CAPTURED_AGENT["tools"]])
print("invoke via:   ", CAPTURED_INVOKE["surface"], "(not threads/runs)")
print("output_text:  ", CAPTURED_INVOKE["output_text"])


agent name:    chinook-desk
kind:          prompt
model:         <a Foundry model deployment>
instructions:  Use the tools. Do not invent numbers or names.
tools:         ['invoice_count', 'invoice_total', 'support_rep']
invoke via:    responses (not threads/runs)
output_text:   Helena Holy has 7 invoices totaling $49.62. Her support representative is Steve Johnson.


### Map each field back to a cell you already wrote

The managed product did not invent a new idea. It stored the same four things and hid the `for`.


In [3]:
rows = [
    ("name", "A label for the desk. Module 08 called it desk_tools."),
    ("kind = prompt", "A prompt agent is instructions plus tools. A hosted agent is your container. We stay on prompt."),
    ("model", "os.environ['MODEL_DEFAULT'], now a Foundry deployment name."),
    ("instructions", "The system message. Module 00."),
    ("tools", "The JSON you typed in 02. Foundry can also attach MCP servers here."),
    ("conversation + responses", "The messages list, plus one create. The for-loop is their runtime."),
    ("output_text", "The last assistant sentence. Helena is still 7 / $49.62 / Steve Johnson."),
]
print(f"{'field':28} what you already have")
print("-" * 88)
for field, meaning in rows:
    print(f"{field:28} {meaning}")


field                        what you already have
----------------------------------------------------------------------------------------
name                         A label for the desk. Module 08 called it desk_tools.
kind = prompt                A prompt agent is instructions plus tools. A hosted agent is your container. We stay on prompt.
model                        os.environ['MODEL_DEFAULT'], now a Foundry deployment name.
instructions                 The system message. Module 00.
tools                        The JSON you typed in 02. Foundry can also attach MCP servers here.
conversation + responses     The messages list, plus one create. The for-loop is their runtime.
output_text                  The last assistant sentence. Helena is still 7 / $49.62 / Steve Johnson.


### Optional: a provisioned project

If the two Foundry names in `.env` are empty, stop here. That is the designed path.

If the instructor filled them, the live walk is the **portal**, not a second SDK in this venv. Adding `azure-ai-projects` would pull another client into the shared environment that 00–15 already run on. We do not do that.

In the portal, the instructor shows four things and sits down:

1. Agent Builder — the no-code surface for the same definition you just printed.
2. A Connected Agents wiring — module 13's desk, as a feature.
3. The trace view — module 15's messages list, as a timeline.
4. Content filters / prompt shields — a hosted version of the scan in module 14. Still incomplete. Still necessary.


In [4]:
if foundry_endpoint:
    print("A Foundry endpoint is set. The live walk is the portal, not this cell.")
    print("Host only:", urlparse(foundry_endpoint).netloc)
else:
    print("No Foundry endpoint. Designed path: the captured shape is the lesson.")


No Foundry endpoint. Designed path: the captured shape is the lesson.


## 3. Observe

Same Chinook question. Same three facts. What you can no longer see, once the runtime is theirs.


In [5]:
hidden = [
    ("the for-loop", "orchestration", "Their runtime. You can still cap it if they expose a turn limit."),
    ("the tool dispatch", "tools", "Portable if the tools are MCP. Sticky if they only exist in the catalogue."),
    ("the messages list", "context / traces", "You get a timeline instead of a print. Redact before you turn it on."),
    ("who the agent runs as", "identity", "Entra Agent ID, not the class OpenAI key. This is the sticky one."),
    ("the dollar ledger", "governance", "Module 15 printed it. A platform will too, on their invoice cycle."),
]
print(f"{'what you printed by hand':28} {'layer':18} what a platform does with it")
print("-" * 100)
for what, layer, note in hidden:
    print(f"{what:28} {layer:18} {note}")

print()
print("portable:", "model, MCP tools, an open graph")
print("sticky:  ", "memory, identity, the audit trail")


what you printed by hand     layer              what a platform does with it
----------------------------------------------------------------------------------------------------
the for-loop                 orchestration      Their runtime. You can still cap it if they expose a turn limit.
the tool dispatch            tools              Portable if the tools are MCP. Sticky if they only exist in the catalogue.
the messages list            context / traces   You get a timeline instead of a print. Redact before you turn it on.
who the agent runs as        identity           Entra Agent ID, not the class OpenAI key. This is the sticky one.
the dollar ledger            governance         Module 15 printed it. A platform will too, on their invoice cycle.

portable: model, MCP tools, an open graph
sticky:   memory, identity, the audit trail


## 4. Challenge

Two short classifications. The words mean the same thing they did in Learn.

**Layers.** For each row, bind `"portable"` or `"sticky"`.

**Buying.** For each job, bind `"local"` (you keep the loop you wrote), `"foundry"` (you are buying execute / identity / filters), or `"neither"` (it should not be an agent).

A couple of rows have more than one defensible answer. The assert only checks the ones that do not.

| key | What you are looking at |
|---|---|
| `layer["model"]` | Swap the chat model next quarter and re-run evals. |
| `layer["mcp_tools"]` | `get_fact` / `get_flight` behind the server from module 06. |
| `layer["open_graph"]` | The LangGraph loop from module 09, still in this repo. |
| `layer["vector_memory"]` | A year of ticket embeddings in a vendor store, with their filters. |
| `layer["identity"]` | The agent must run as itself in Entra, not on the class key. |
| `layer["hosted_traces"]` | Eighteen months of prompts and tool args in a vendor timeline. |
| `buy["nightly_csv"]` | Export yesterday's invoices to a file share, same steps every night. |
| `buy["helena_desk"]` | The three Chinook lookups you already run in 08, 09 and 15. |
| `buy["refunds_with_identity"]` | Customer-facing refunds. Risk wants an agent identity and a content filter. |


In [ ]:
layer = {}
buy = {}

# layer["model"] = ...
# layer["mcp_tools"] = ...
# layer["open_graph"] = ...
# layer["vector_memory"] = ...
# layer["identity"] = ...
# layer["hosted_traces"] = ...

# buy["nightly_csv"] = ...
# buy["helena_desk"] = ...
# buy["refunds_with_identity"] = ...

In [ ]:
allowed_layer = {"portable", "sticky"}
allowed_buy = {"local", "foundry", "neither"}

assert set(layer) == {
    "model", "mcp_tools", "open_graph",
    "vector_memory", "identity", "hosted_traces",
}, "bind every layer key"
assert set(buy) == {"nightly_csv", "helena_desk", "refunds_with_identity"}, "bind every buy key"
assert set(layer.values()) <= allowed_layer, "each layer value is portable or sticky"
assert set(buy.values()) <= allowed_buy, "each buy value is local, foundry, or neither"

assert layer["model"] == "portable", "a compatible model is a config change"
assert layer["mcp_tools"] == "portable", "MCP is the portability you already built"
assert layer["open_graph"] == "portable", "an open graph still runs here"
assert layer["identity"] == "sticky", "identity is where lock-in lives"
assert layer["hosted_traces"] == "sticky", "the audit trail is a project to move"
assert buy["nightly_csv"] == "neither", "known steps, every night, is a workflow"

print("layer:")
for k, v in layer.items():
    print(f"  {k:18} {v}")
print("buy:")
for k, v in buy.items():
    print(f"  {k:24} {v}")
